# 🌱 Weed Detection: YOLO11 trained on HuggingFace + TiHAN Paddy (merged)

This notebook trains **one YOLO11 model** on two datasets:

| Dataset | Source | Annotation | Becomes |
|---|---|---|---|
| **Weed-Crop Aerial** | HuggingFace `Francesco/weed-crop-aerial` | COCO boxes: `crop`, `weed` | classes `0 crop`, `1 weed` |
| **TiHAN Paddy** | your `Paddy_Dataset.zip` | **points**: one `x y` pixel per rice **panicle** (grain head) | class `2 panicle` (box drawn around each point) |

> ⚠️ **Why a third class?** The TiHAN labels are *not* YOLO boxes. Each line is the pixel position of a
> rice panicle. They contain no weeds. Labelling them `weed` would teach the drone to spray rice, so they get
> their own `panicle` class. The project treats `panicle` as a crop, so panicles are protected from
> spraying. You can switch to `TIHAN_CLASS = "crop"` in the config cell instead.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. Training checkpoints are written to Google Drive, so a
disconnect does not lose your progress (see *Resume*).

**Before you start:** upload `Paddy_Dataset.zip` to Google Drive at `MyDrive/datasets/Paddy_Dataset.zip`
(or drag it into the Colab *Files* panel).

In [ ]:
!nvidia-smi
!pip install -q ultralytics huggingface_hub pyyaml

In [ ]:
import torch, ultralytics
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Ultralytics:", ultralytics.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1 · Configuration
Everything you may want to change is in this cell.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- datasets
USE_HF_DATASET = True                      # Francesco/weed-crop-aerial  (crop + weed boxes)
HF_REPO = "Francesco/weed-crop-aerial"

USE_TIHAN_DATASET = True                   # TiHAN Paddy dataset (zip)
TIHAN_ZIP_CANDIDATES = [                   # the first path that exists is used
    "/content/drive/MyDrive/datasets/Paddy_Dataset.zip",
    "/content/drive/MyDrive/Paddy_Dataset.zip",
    "/content/Paddy_Dataset.zip",
]
# TiHAN labels are POINTS (one "x y" pixel per rice panicle), not boxes:
#   "panicle" -> new 3rd class (recommended, keeps crop/weed meaning intact)
#   "crop"    -> every panicle point becomes a crop box
TIHAN_CLASS = "panicle"
POINT_BOX_PX = 64            # side of the square box drawn around each point (pixels, images are 1150x850)
TIHAN_VAL_FRACTION = 0.15    # validation share, split by source image so patches never leak

# ---------------------------------------------------------------- output
WORK = Path("/content")
MERGED = WORK / "merged_yolo"                                   # merged YOLO dataset
DRIVE_DIR = Path("/content/drive/MyDrive/AI-Weed-Detection")    # runs + final model are kept here

# ---------------------------------------------------------------- training
BASE_MODEL = "yolo11s.pt"
EPOCHS = 80
IMGSZ = 640
BATCH = 16           # safe for yolo11s @640 on a T4; -1 = auto
PATIENCE = 20
SEED = 42
RUN_NAME = "weed_yolo11s_merged"

assert TIHAN_CLASS in ("panicle", "crop")
CLASS_NAMES = ["crop", "weed"] + (["panicle"] if USE_TIHAN_DATASET and TIHAN_CLASS == "panicle" else [])
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
print("Classes:", CLASS_TO_ID)

In [ ]:
# Google Drive keeps the dataset zip, the training checkpoints and the final model
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - Google Drive not mounted")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

import shutil
if MERGED.exists():
    shutil.rmtree(MERGED)          # always rebuild the merged dataset from scratch
for sub in ("images", "labels"):
    (MERGED / sub).mkdir(parents=True, exist_ok=True)
print("Merged dataset will be built in", MERGED)

## 2 · HuggingFace Weed-Crop Aerial dataset (COCO → YOLO)

In [ ]:
if USE_HF_DATASET:
    from huggingface_hub import hf_hub_download
    import tarfile

    archive_path = hf_hub_download(repo_id=HF_REPO, filename="dataset.tar.gz", repo_type="dataset")
    HF_RAW = WORK / "weed_dataset"
    if HF_RAW.exists():
        shutil.rmtree(HF_RAW)
    HF_RAW.mkdir(parents=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(path=HF_RAW, filter="data")

    # The archive nests the data under a long path - locate the split folders automatically.
    annotation_files = sorted(HF_RAW.rglob("_annotations.coco.json"))
    assert annotation_files, "No _annotations.coco.json found in the HuggingFace archive"
    HF_ROOT = annotation_files[0].parent.parent
    print("HuggingFace dataset root:", HF_ROOT)
    print("Splits:", sorted(p.name for p in HF_ROOT.iterdir() if p.is_dir()))

In [ ]:
import json
from collections import defaultdict

def convert_coco_split(split_dir, out_split, prefix="hf_"):
    """COCO json -> YOLO txt. Keeps only crop / weed and writes them with the merged class ids."""
    data = json.loads((split_dir / "_annotations.coco.json").read_text())
    category_map = {c["id"]: CLASS_TO_ID[c["name"]] for c in data["categories"] if c["name"] in ("crop", "weed")}
    assert set(category_map.values()) == {CLASS_TO_ID["crop"], CLASS_TO_ID["weed"]}, \
        f"crop/weed categories not found in {split_dir}: {data['categories']}"

    anns = defaultdict(list)
    for a in data["annotations"]:
        if a["category_id"] in category_map:
            anns[a["image_id"]].append(a)

    img_out = MERGED / "images" / out_split
    lab_out = MERGED / "labels" / out_split
    img_out.mkdir(parents=True, exist_ok=True)
    lab_out.mkdir(parents=True, exist_ok=True)

    n_img = n_box = 0
    for info in data["images"]:
        name = Path(info["file_name"]).name
        src = split_dir / info["file_name"]
        if not src.exists():
            src = split_dir / name
        if not src.exists():
            hits = list(split_dir.rglob(name))
            assert hits, f"Image not found: {info['file_name']}"
            src = hits[0]
        w, h = info["width"], info["height"]
        lines = []
        for a in anns[info["id"]]:
            x, y, bw, bh = a["bbox"]
            x1, y1 = max(0.0, min(w, x)), max(0.0, min(h, y))
            x2, y2 = max(0.0, min(w, x + bw)), max(0.0, min(h, y + bh))
            if x2 - x1 <= 0 or y2 - y1 <= 0:
                continue
            lines.append(f"{category_map[a['category_id']]} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} "
                         f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
        stem = prefix + Path(name).stem
        shutil.copy2(src, img_out / f"{stem}{Path(name).suffix}")
        (lab_out / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        n_img += 1
        n_box += len(lines)
    print(f"  HF {split_dir.name:>5} -> {out_split:<10} {n_img:5d} images {n_box:6d} boxes")

if USE_HF_DATASET:
    for split, out_split in {"train": "train", "valid": "val", "test": "test_hf"}.items():
        convert_coco_split(HF_ROOT / split, out_split)

## 3 · TiHAN Paddy dataset (points → boxes)

Zip layout: `Paddy_Dataset/{train,test}/{images,labels}`. The images are 1150 × 850 patches named
`IMG_<n>_patch_<r>_<c>.jpg`. Each label line is `x y`, the pixel position of one rice panicle.

In [ ]:
import zipfile

if USE_TIHAN_DATASET:
    src = next((Path(p) for p in TIHAN_ZIP_CANDIDATES if Path(p).exists()), None)
    if src is None and IN_COLAB:
        from google.colab import files
        print("Paddy_Dataset.zip not found on Drive - pick it to upload (slower than Drive for 290 MB)")
        uploaded = files.upload()
        src = WORK / next(iter(uploaded))
    assert src is not None and src.exists(), f"Paddy_Dataset.zip not found in {TIHAN_ZIP_CANDIDATES}"

    local_zip = WORK / "Paddy_Dataset.zip"
    if src.resolve() != local_zip.resolve():
        print("Copying", src, "to local disk ...")
        shutil.copy2(src, local_zip)             # unzipping from local disk is much faster than from Drive

    TIHAN_RAW = WORK / "tihan_raw"
    if TIHAN_RAW.exists():
        shutil.rmtree(TIHAN_RAW)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(TIHAN_RAW)

    TIHAN_ROOT = next(p.parent for p in TIHAN_RAW.rglob("train") if (p / "images").is_dir())
    for split in ("train", "test"):
        n_img = len(list((TIHAN_ROOT / split / "images").glob("*.jpg")))
        n_lab = len(list((TIHAN_ROOT / split / "labels").glob("*.txt")))
        print(f"TiHAN {split}: {n_img} images, {n_lab} label files")

In [ ]:
# Inspect the label format before converting anything
from collections import Counter
from PIL import Image

if USE_TIHAN_DATASET:
    tokens = Counter()
    for lab in list((TIHAN_ROOT / "train" / "labels").glob("*.txt"))[:300]:
        for line in lab.read_text().splitlines():
            if line.strip():
                tokens[len(line.split())] += 1
    sample = next((TIHAN_ROOT / "train" / "images").glob("*.jpg"))
    print("Values per label line:", dict(tokens), "| image size:", Image.open(sample).size)
    print("Example label:", (TIHAN_ROOT / "train" / "labels" / f"{sample.stem}.txt").read_text().splitlines()[:3])
    TIHAN_FORMAT = "points" if tokens.most_common(1)[0][0] == 2 else "yolo"
    assert TIHAN_FORMAT == "points" or tokens.most_common(1)[0][0] == 5, f"Unknown label format: {tokens}"
    print("Detected format:", TIHAN_FORMAT)

In [ ]:
import random

def tihan_label_lines(label_path, w, h, cls_id):
    """Point "x y" (pixels) -> square box of POINT_BOX_PX. 5-column YOLO lines are kept (class remapped)."""
    lines = []
    for raw in label_path.read_text().splitlines():
        p = raw.split()
        if len(p) == 5:                                  # already YOLO
            lines.append(f"{cls_id} " + " ".join(p[1:]))
            continue
        if len(p) != 2:
            continue
        x, y = float(p[0]), float(p[1])
        half = POINT_BOX_PX / 2
        x1, y1 = max(0.0, x - half), max(0.0, y - half)
        x2, y2 = min(float(w), x + half), min(float(h), y + half)
        if x2 - x1 < 2 or y2 - y1 < 2:                   # point outside / on the edge
            continue
        lines.append(f"{cls_id} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} "
                     f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
    return lines

def copy_tihan(images, out_split, cls_id, prefix="tihan_"):
    img_out = MERGED / "images" / out_split
    lab_out = MERGED / "labels" / out_split
    img_out.mkdir(parents=True, exist_ok=True)
    lab_out.mkdir(parents=True, exist_ok=True)
    n_box = skipped = 0
    for img in images:
        lab = img.parent.parent / "labels" / f"{img.stem}.txt"
        if not lab.exists():
            skipped += 1
            continue
        w, h = Image.open(img).size
        lines = tihan_label_lines(lab, w, h, cls_id)
        shutil.copy2(img, img_out / f"{prefix}{img.name}")
        (lab_out / f"{prefix}{img.stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        n_box += len(lines)
    print(f"  TiHAN -> {out_split:<10} {len(images) - skipped:5d} images {n_box:6d} boxes"
          + (f"  ({skipped} without label skipped)" if skipped else ""))

if USE_TIHAN_DATASET:
    cls_id = CLASS_TO_ID[TIHAN_CLASS]
    train_imgs = sorted((TIHAN_ROOT / "train" / "images").glob("*.jpg"))
    test_imgs = sorted((TIHAN_ROOT / "test" / "images").glob("*.jpg"))

    # Patches of the same field photo (IMG_12_patch_*) overlap visually - keep them in the same split.
    groups = defaultdict(list)
    for img in train_imgs:
        groups[img.stem.split("_patch_")[0]].append(img)
    keys = sorted(groups)
    random.Random(SEED).shuffle(keys)
    val_imgs, target = [], TIHAN_VAL_FRACTION * len(train_imgs)
    while keys and len(val_imgs) < target:
        val_imgs += groups[keys.pop()]
    tr_imgs = [img for k in keys for img in groups[k]]

    copy_tihan(tr_imgs, "train", cls_id)
    copy_tihan(val_imgs, "val", cls_id)
    copy_tihan(test_imgs, "test_tihan", cls_id)
    print(f"Point boxes: {POINT_BOX_PX}x{POINT_BOX_PX}px, class {cls_id} = '{TIHAN_CLASS}'")

In [ ]:
# Visual check: the generated boxes must sit on the rice panicles
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show_yolo(img_path, ax):
    im = Image.open(img_path)
    w, h = im.size
    ax.imshow(im)
    lab = MERGED / "labels" / img_path.parent.name / f"{img_path.stem}.txt"
    colors = {CLASS_TO_ID["crop"]: "#20a7a0", CLASS_TO_ID["weed"]: "#ee5f4c"}
    for line in lab.read_text().splitlines():
        c, cx, cy, bw, bh = map(float, line.split())
        ax.add_patch(patches.Rectangle(((cx - bw / 2) * w, (cy - bh / 2) * h), bw * w, bh * h,
                                       fill=False, lw=1.5, ec=colors.get(int(c), "#ff00ff")))
    ax.set_title(f"{img_path.name}\n{len(lab.read_text().splitlines())} boxes", fontsize=9)
    ax.axis("off")

samples = (sorted((MERGED / "images" / "train").glob("tihan_*"))[:2]
           + sorted((MERGED / "images" / "train").glob("hf_*"))[:2])
if samples:
    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 4.5))
    for ax, p in zip(axes if len(samples) > 1 else [axes], samples):
        show_yolo(p, ax)
    plt.tight_layout()
    plt.show()

## 4 · Merged dataset: `data.yaml`, statistics and sanity checks

In [ ]:
import yaml
import pandas as pd

def nonempty(split):
    d = MERGED / "images" / split
    return d.is_dir() and any(d.iterdir())

test_splits = [f"images/{s}" for s in ("test_hf", "test_tihan") if nonempty(s)]
names = {i: n for i, n in enumerate(CLASS_NAMES)}
config = {"path": str(MERGED), "train": "images/train", "val": "images/val",
          "test": test_splits if len(test_splits) > 1 else test_splits[0], "names": names}
(MERGED / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False))

# one yaml per source, to measure each dataset separately after training
for s in ("test_hf", "test_tihan"):
    if nonempty(s):
        (MERGED / f"data_{s}.yaml").write_text(yaml.safe_dump({**config, "test": f"images/{s}"}, sort_keys=False))

print((MERGED / "data.yaml").read_text())

rows = []
for split_dir in sorted((MERGED / "images").iterdir()):
    split = split_dir.name
    labels = list((MERGED / "labels" / split).glob("*.txt"))
    images = list(split_dir.iterdir())
    assert len(images) == len(labels), f"{split}: {len(images)} images vs {len(labels)} labels"
    counts = Counter()
    for lab in labels:
        for line in lab.read_text().splitlines():
            v = line.split()
            assert len(v) == 5, f"bad line in {lab}: {line}"
            c, cx, cy, bw, bh = map(float, v)
            assert int(c) in names and all(0 <= t <= 1 for t in (cx, cy, bw, bh)) and bw > 0 and bh > 0, lab
            counts[names[int(c)]] += 1
    row = {"split": split, "images": len(images),
           "hf images": sum(p.name.startswith("hf_") for p in images),
           "tihan images": sum(p.name.startswith("tihan_") for p in images)}
    row.update({f"{n} boxes": counts[n] for n in CLASS_NAMES})
    rows.append(row)

display(pd.DataFrame(rows).set_index("split"))
print("All label checks passed.")

## 5 · Train YOLO11

Checkpoints go to `MyDrive/AI-Weed-Detection/runs/`. **If Colab disconnects**, re-run the cells above
(they rebuild the dataset in about 2 minutes), set `RESUME = True` and run this cell again.

In [ ]:
from ultralytics import YOLO

RESUME = False
RUNS = DRIVE_DIR / "runs"

if RESUME:
    last = max(RUNS.glob(f"{RUN_NAME}*/weights/last.pt"), key=lambda p: p.stat().st_mtime)
    print("Resuming from", last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(MERGED / "data.yaml"),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=2,
        optimizer="auto",
        cos_lr=True,
        close_mosaic=10,
        project=str(RUNS),
        name=RUN_NAME,
        plots=True,
        seed=SEED,
    )

RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
print("Run folder:", RUN_DIR, "| best.pt exists:", BEST.exists())

In [ ]:
from IPython.display import Image as IPImage
for name in ("results.png", "confusion_matrix_normalized.png", "labels.jpg"):
    f = RUN_DIR / name
    if f.exists():
        print(name)
        display(IPImage(filename=str(f), width=900))

## 6 · Evaluate: overall and per dataset

The per-dataset rows show whether adding TiHAN changed weed detection on the original HuggingFace test set.
If your previous model is on Drive at `AI-Weed-Detection/models/best.pt`, it is evaluated too for comparison.

In [ ]:
def evaluate(weights, data_yaml, label):
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml), split="test", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False,
              device=0 if torch.cuda.is_available() else "cpu", project=str(WORK / "eval"), name=label)
    row = {"model / test set": label, "precision": r.box.mp, "recall": r.box.mr,
           "mAP50": r.box.map50, "mAP50-95": r.box.map}
    for i, ap in zip(r.box.ap_class_index, r.box.ap50):
        row[f"AP50 {r.names[int(i)]}"] = ap
    return row

results = [evaluate(BEST, MERGED / "data.yaml", "merged · all test")]
for s in ("test_hf", "test_tihan"):
    y = MERGED / f"data_{s}.yaml"
    if y.exists():
        results.append(evaluate(BEST, y, f"merged · {s}"))

OLD_MODEL = DRIVE_DIR / "models" / "best.pt"          # the HuggingFace-only model from the previous notebook
if OLD_MODEL.exists() and (MERGED / "data_test_hf.yaml").exists():
    try:
        results.append(evaluate(OLD_MODEL, MERGED / "data_test_hf.yaml", "previous · test_hf"))
    except Exception as exc:
        print("Could not evaluate previous model:", exc)

display(pd.DataFrame(results).set_index("model / test set").round(3))

In [ ]:
# Predictions on unseen test images (HF and TiHAN)
best = YOLO(str(BEST))
picks = (sorted((MERGED / "images" / "test_hf").glob("*"))[:2] if nonempty("test_hf") else []) + \
        (sorted((MERGED / "images" / "test_tihan").glob("*"))[:2] if nonempty("test_tihan") else [])
fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 4.5))
for ax, p in zip(axes if len(picks) > 1 else [axes], picks):
    r = best.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(f"{p.name}\n{len(r.boxes)} detections", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7 · Save the model (Drive + download)

In [ ]:
import json, time

models_dir = DRIVE_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
final = models_dir / "best_merged.pt"           # does not overwrite your previous best.pt
shutil.copy2(BEST, final)
(models_dir / "best_merged_info.json").write_text(json.dumps({
    "classes": CLASS_NAMES, "datasets": {"huggingface": USE_HF_DATASET, "tihan_paddy": USE_TIHAN_DATASET},
    "tihan_class": TIHAN_CLASS, "point_box_px": POINT_BOX_PX, "base_model": BASE_MODEL, "epochs": EPOCHS,
    "imgsz": IMGSZ, "trained": time.strftime("%Y-%m-%d %H:%M"), "metrics": results}, indent=2, default=float))
print("Saved:", final, f"({final.stat().st_size / 1e6:.1f} MB)")

EXPORT_NCNN = False      # True -> also build the Raspberry Pi model (models/best_ncnn_model)
if EXPORT_NCNN:
    out = YOLO(str(BEST)).export(format="ncnn", imgsz=IMGSZ)
    shutil.make_archive(str(models_dir / "best_merged_ncnn_model"), "zip", out)
    print("NCNN model:", models_dir / "best_merged_ncnn_model.zip")

if IN_COLAB:
    from google.colab import files
    files.download(str(final))

## 8 · Use the model in the project

1. Copy the downloaded **`best_merged.pt`** into the project's `models/` folder and rename it to **`best.pt`**.
   Keep the old file as `models/best_v1.pt` if you want to compare.
2. `configs/drone_config.yaml` already lists `panicle` as a crop, so the drone never sprays rice panicles:
   ```yaml
   detector:
     weed_classes: [weed]
     crop_classes: [crop, panicle]
   ```
3. Refresh the results shown in the website and simulator:
   ```bash
   python src/batch_detect.py
   python dashboard/scripts/build_showcase.py
   python simulation/record_demo.py
   python -m unittest discover tests
   ```